In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create output directory
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

# ============================================================================
# STEP 1: Load Data
# ============================================================================
print("Loading data...")

# YOU WILL PROVIDE THIS PATH
df= pd.read_csv("data/processed_rat_data.csv")


Loading data...


In [3]:

print(f"Data loaded: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

# ============================================================================
# STEP 2: Prepare Data
# ============================================================================
print("\n" + "="*80)
print("STEP 2: Data Preparation")
print("="*80)

# Create group_short if needed
def shorten_group_name(name):
    if 'Group 1' in name and 'RO water with < 20' in name:
        return 'G01: RO <20 TDS'
    elif 'Group 2' in name:
        return 'G02: RO 50-75 TDS'
    elif 'Group 3' in name and 'alternate' not in name.lower():
        return 'G03: RO 125-150 TDS'
    elif 'Group 4' in name:
        return 'G04: Telugu Ganga'
    elif 'Group 5' in name and 'alternate' not in name.lower():
        return 'G05: Kalyani Dam'
    elif 'Group 6' in name and 'alternate' not in name.lower():
        return 'G06: Ground Water'
    elif 'Group 7' in name:
        return 'G07: RO <20 (Fasting)'
    elif 'Group 8' in name:
        return 'G08: Kalyani (Fasting)'
    elif 'Group 9' in name:
        return 'G09: BIS Standard'
    elif 'Group 10' in name:
        return 'G10: Ground (Fasting)'
    elif 'Group 11' in name:
        return 'G11: RO 125-150 (Fasting)'
    return name

if 'group_short' not in df.columns:
    df['group_short'] = df['water_group'].apply(shorten_group_name)

# Get final weight data (week 15 only) OR calculate total weight gain
if 'final_body_weight' in df.columns and 'initial_body_weight' in df.columns:
    # Option 1: Use existing columns
    final_data = df[df['week'] == 15].copy()
    final_data['total_weight_gain'] = final_data['final_body_weight'] - final_data['initial_body_weight']
    print("Using existing final_body_weight column")
elif 'cumulative_weight_gain' in df.columns:
    # Option 2: Use cumulative weight gain at week 15
    final_data = df[df['week'] == 15].copy()
    final_data['total_weight_gain'] = final_data['cumulative_weight_gain']
    print("Using cumulative_weight_gain column")
else:
    # Option 3: Calculate from weekly_weight_gain
    print("Calculating total weight gain from weekly data...")
    weight_gain_by_rat = df.groupby('unique_rat_id').agg({
        'weekly_weight_gain': 'sum',
        'gender': 'first',
        'group_short': 'first',
        'water_group': 'first',
        'initial_body_weight': 'first'
    }).reset_index()
    weight_gain_by_rat.rename(columns={'weekly_weight_gain': 'total_weight_gain'}, inplace=True)
    final_data = weight_gain_by_rat

print(f"\nRats in analysis: {len(final_data)}")
print(f"Groups: {sorted(final_data['group_short'].unique())}")

# ============================================================================
# STEP 3: Calculate Summary Statistics
# ============================================================================
print("\n" + "="*80)
print("STEP 3: Summary Statistics by Group and Gender")
print("="*80)

summary_stats = final_data.groupby(['group_short', 'gender'])['total_weight_gain'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max', 
    ('q25', lambda x: x.quantile(0.25)),
    ('q75', lambda x: x.quantile(0.75))
]).reset_index()

print("\n--- Weight Gain Summary ---")
print(summary_stats.to_string(index=False))

# Save summary
summary_stats.to_csv(output_dir / 'weight_gain_distribution_summary.csv', index=False)
print(f"\n✓ Saved: weight_gain_distribution_summary.csv")

# Overall statistics by gender
print("\n--- Overall by Gender ---")
gender_summary = final_data.groupby('gender')['total_weight_gain'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
])
print(gender_summary)

# Overall statistics by group
print("\n--- Overall by Water Group ---")
group_summary = final_data.groupby('group_short')['total_weight_gain'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
])
print(group_summary.sort_values('median', ascending=False))

# ============================================================================
# STEP 4: Statistical Tests
# ============================================================================
print("\n" + "="*80)
print("STEP 4: Statistical Analysis")
print("="*80)

# Test gender differences
male_gains = final_data[final_data['gender'] == 'male']['total_weight_gain']
female_gains = final_data[final_data['gender'] == 'female']['total_weight_gain']

t_stat, p_val = stats.ttest_ind(male_gains, female_gains)
print(f"\nGender comparison (t-test):")
print(f"  Male mean: {male_gains.mean():.2f} g")
print(f"  Female mean: {female_gains.mean():.2f} g")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value: {p_val:.4f}")
if p_val < 0.05:
    print("  → Significant gender difference")
else:
    print("  → No significant gender difference")

# ANOVA across groups
print("\n--- ANOVA: Weight gain across water groups ---")
groups_data = [final_data[final_data['group_short'] == g]['total_weight_gain'].values 
               for g in sorted(final_data['group_short'].unique())]
f_stat, p_val_anova = stats.f_oneway(*groups_data)
print(f"  F-statistic: {f_stat:.3f}")
print(f"  p-value: {p_val_anova:.4f}")
if p_val_anova < 0.05:
    print("  → Significant differences among water groups")
else:
    print("  → No significant differences among water groups")

# ============================================================================
# STEP 5: Create Visualizations (3 panels only)
# ============================================================================
print("\n" + "="*80)
print("STEP 5: Creating Visualizations")
print("="*80)

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

groups_ordered = sorted(final_data['group_short'].unique())

# ============================================================================
# PANEL 1: Violin Plot - Gender Comparison
# ============================================================================
print("\nCreating Panel 1: Violin plot by gender...")
ax1 = fig.add_subplot(gs[0, :])

# Create violin plot for each gender
parts = ax1.violinplot([male_gains, female_gains],
                       positions=[1, 2],
                       showmeans=True,
                       showmedians=True,
                       showextrema=True,
                       widths=0.7)

# Color the violins
colors = ['#2E86AB', '#E63946']
for pc, color in zip(parts['bodies'], colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.6)

ax1.set_xticks([1, 2])
ax1.set_xticklabels(['Male', 'Female'], fontsize=13)
ax1.set_ylabel('Total Weight Gain (g)', fontsize=13, fontweight='bold')
ax1.set_title('Weight Gain Distribution by Gender', fontsize=15, fontweight='bold', pad=15)
ax1.grid(True, alpha=0.3, axis='y')

# Add statistics text
ax1.text(1, male_gains.max() + 10, 
         f'Mean: {male_gains.mean():.1f}g\nMedian: {male_gains.median():.1f}g\nn={len(male_gains)}',
         ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
ax1.text(2, female_gains.max() + 10,
         f'Mean: {female_gains.mean():.1f}g\nMedian: {female_gains.median():.1f}g\nn={len(female_gains)}',
         ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

# ============================================================================
# PANEL 2: Bar Chart - Mean Weight Gain by Group
# ============================================================================
print("Creating Panel 2: Mean comparison bar chart...")
ax2 = fig.add_subplot(gs[1, 0])

# Get means and SEMs
male_summary = summary_stats[summary_stats['gender'] == 'male'].sort_values('group_short')
female_summary = summary_stats[summary_stats['gender'] == 'female'].sort_values('group_short')

x = np.arange(len(groups_ordered))
width = 0.35

# Calculate SEM
male_sem = male_summary['std'] / np.sqrt(male_summary['count'])
female_sem = female_summary['std'] / np.sqrt(female_summary['count'])

bars1 = ax2.bar(x - width/2, male_summary['mean'].values, width,
                yerr=male_sem.values,
                label='Male',
                color='#2E86AB',
                alpha=0.7,
                capsize=3)

bars2 = ax2.bar(x + width/2, female_summary['mean'].values, width,
                yerr=female_sem.values,
                label='Female',
                color='#E63946',
                alpha=0.7,
                capsize=3)

ax2.set_ylabel('Mean Total Weight Gain (g)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Water Group', fontsize=12, fontweight='bold')
ax2.set_title('Mean Weight Gain by Water Group', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(groups_ordered, rotation=45, ha='right', fontsize=9)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

# ============================================================================
# PANEL 3: Scatter - Initial Weight vs Weight Gain
# ============================================================================
print("Creating Panel 3: Initial weight vs gain scatter...")
ax3 = fig.add_subplot(gs[1, 1])

male_data = final_data[final_data['gender'] == 'male']
female_data = final_data[final_data['gender'] == 'female']

ax3.scatter(male_data['initial_body_weight'], male_data['total_weight_gain'],
           alpha=0.6, s=60, c='#2E86AB', label='Male', edgecolors='black', linewidth=0.5)
ax3.scatter(female_data['initial_body_weight'], female_data['total_weight_gain'],
           alpha=0.6, s=60, c='#E63946', label='Female', edgecolors='black', linewidth=0.5)

# Add trend lines
if len(male_data) > 1:
    z_m = np.polyfit(male_data['initial_body_weight'], male_data['total_weight_gain'], 1)
    p_m = np.poly1d(z_m)
    ax3.plot(male_data['initial_body_weight'].sort_values(), 
             p_m(male_data['initial_body_weight'].sort_values()),
             '--', color='#2E86AB', alpha=0.5, linewidth=2)

if len(female_data) > 1:
    z_f = np.polyfit(female_data['initial_body_weight'], female_data['total_weight_gain'], 1)
    p_f = np.poly1d(z_f)
    ax3.plot(female_data['initial_body_weight'].sort_values(),
             p_f(female_data['initial_body_weight'].sort_values()),
             '--', color='#E63946', alpha=0.5, linewidth=2)

ax3.set_xlabel('Initial Body Weight (g)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Total Weight Gain (g)', fontsize=12, fontweight='bold')
ax3.set_title('Initial Weight vs Total Weight Gain', fontsize=13, fontweight='bold')
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3)

# Calculate correlations
if len(male_data) > 1:
    corr_m = male_data['initial_body_weight'].corr(male_data['total_weight_gain'])
    ax3.text(0.05, 0.95, f'Male r={corr_m:.2f}', transform=ax3.transAxes,
             fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='#2E86AB', alpha=0.3))

if len(female_data) > 1:
    corr_f = female_data['initial_body_weight'].corr(female_data['total_weight_gain'])
    ax3.text(0.05, 0.85, f'Female r={corr_f:.2f}', transform=ax3.transAxes,
             fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='#E63946', alpha=0.3))

# ============================================================================
# Save Figure
# ============================================================================
plt.suptitle('Weight Gain Distribution Analysis',
             fontsize=17, fontweight='bold', y=0.995)

output_path = output_dir / 'weight_gain_distribution_analysis.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ Saved: {output_path}")
plt.close()

# ============================================================================
# STEP 6: Best/Worst Performing Groups
# ============================================================================
print("\n" + "="*80)
print("STEP 6: Best and Worst Performing Groups")
print("="*80)

print("\n--- Top 5 Groups by Mean Weight Gain ---")
top_groups = group_summary.sort_values('mean', ascending=False).head()
print(top_groups[['mean', 'median', 'count']])

print("\n--- Bottom 5 Groups by Mean Weight Gain ---")
bottom_groups = group_summary.sort_values('mean').head()
print(bottom_groups[['mean', 'median', 'count']])

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nGenerated files:")
print("  1. weight_gain_distribution_analysis.png - Main visualization (3 panels)")
print("  2. weight_gain_distribution_summary.csv - Detailed statistics")
print("\n" + "="*80)

Data loaded: 1650 rows
Columns: ['water_group', 'rat_id', 'rat_number', 'gender', 'week', 'weekly_feed_intake', 'initial_body_weight', 'weekly_weight_gain', 'unique_rat_id', 'cumulative_weight_gain', 'current_body_weight', 'final_body_weight']

STEP 2: Data Preparation
Using existing final_body_weight column

Rats in analysis: 110
Groups: ['G01: RO <20 TDS', 'G02: RO 50-75 TDS', 'G03: RO 125-150 TDS', 'G04: Telugu Ganga', 'G05: Kalyani Dam', 'G06: Ground Water', 'G07: RO <20 (Fasting)', 'G08: Kalyani (Fasting)', 'G09: BIS Standard', 'G10: Ground (Fasting)', 'G11: RO 125-150 (Fasting)']

STEP 3: Summary Statistics by Group and Gender

--- Weight Gain Summary ---
              group_short gender  count  mean  median       std   min   max   q25   q75
          G01: RO <20 TDS female      5 149.6   153.0 17.401149 122.0 165.0 145.0 163.0
          G01: RO <20 TDS   male      5 241.6   237.0 27.682124 212.0 285.0 226.0 248.0
        G02: RO 50-75 TDS female      5 155.2   147.0 13.863621 14